# 🤖 SQL Database AI Agent

A practical **GenAI / Agentic AI** project that converts natural-language questions into SQL, executes read-only SQL against SQLite, and uses an LLM to explain the results.

**Flow:** 👤 User → 🧠 LLM → 📝 SQL → 🛡️ Validation → 🗄️ Database → 📊 Result → 💬 Answer

> ⚠️ Learning project: execution is restricted to read-only `SELECT` queries.

## 🎯 Learning Objectives

- Text-to-SQL with an LLM
- Database schema awareness
- Safe SQL tool execution
- Natural-language result explanation
- Agentic workflow design
- SQL safety and read-only controls

## 🏗️ Architecture

```text
👤 User Question
      ↓
🧠 LLM Agent
      ↓
🔎 Database Schema
      ↓
📝 Generate SQL
      ↓
🛡️ Validate SQL
      ↓
🗄️ Execute SELECT
      ↓
📊 Query Result
      ↓
🧠 LLM Explanation
      ↓
💬 Final Answer
```

In [ ]:
%pip -q install openai

import os, json, re, sqlite3
from getpass import getpass
from openai import OpenAI

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass('Enter your OpenAI API key: ')

client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])
MODEL = 'gpt-4.1-mini'
print('OpenAI client configured successfully.')
print('Model:', MODEL)

## 🗄️ 1. Create a Sample SQLite Database

In [ ]:
DB_PATH = 'sales_ai_agent.db'
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.executescript('''
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS customers;
DROP TABLE IF EXISTS products;

CREATE TABLE customers (customer_id INTEGER PRIMARY KEY, name TEXT, city TEXT, segment TEXT);
CREATE TABLE products (product_id INTEGER PRIMARY KEY, product_name TEXT, category TEXT, price REAL);
CREATE TABLE orders (order_id INTEGER PRIMARY KEY, customer_id INTEGER, product_id INTEGER, quantity INTEGER, order_date TEXT,
    FOREIGN KEY(customer_id) REFERENCES customers(customer_id),
    FOREIGN KEY(product_id) REFERENCES products(product_id));
''')

customers=[(1,'Arun','Chennai','Enterprise'),(2,'Priya','Bengaluru','SMB'),(3,'Rahul','Hyderabad','Enterprise'),(4,'Divya','Chennai','SMB'),(5,'Karthik','Mumbai','Enterprise')]
products=[(101,'Laptop Pro','Computers',85000),(102,'Wireless Mouse','Accessories',1800),(103,'Mechanical Keyboard','Accessories',5500),(104,'Monitor 27','Displays',22000),(105,'USB-C Dock','Accessories',9000)]
orders=[(1001,1,101,2,'2026-01-05'),(1002,1,102,5,'2026-01-12'),(1003,2,103,2,'2026-01-18'),(1004,3,104,3,'2026-02-02'),(1005,4,102,10,'2026-02-10'),(1006,5,105,4,'2026-02-14'),(1007,3,101,1,'2026-03-01'),(1008,2,104,2,'2026-03-07'),(1009,1,105,3,'2026-03-15'),(1010,5,103,6,'2026-03-20')]

cursor.executemany('INSERT INTO customers VALUES (?,?,?,?)',customers)
cursor.executemany('INSERT INTO products VALUES (?,?,?,?)',products)
cursor.executemany('INSERT INTO orders VALUES (?,?,?,?,?)',orders)
conn.commit()
print('Database created:', DB_PATH)

## 🔎 2. Inspect Database Schema

In [ ]:
def get_schema(connection):
    tables=connection.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'").fetchall()
    schema=[]
    for (table,) in tables:
        cols=connection.execute(f'PRAGMA table_info({table})').fetchall()
        schema.append({'table':table,'columns':[{'name':c[1],'type':c[2],'primary_key':bool(c[5])} for c in cols]})
    return schema

schema=get_schema(conn)
print(json.dumps(schema,indent=2))

## 🛡️ 3. Safe Read-Only SQL Tool

In [ ]:
FORBIDDEN_SQL=re.compile(r'\b(INSERT|UPDATE|DELETE|DROP|ALTER|CREATE|REPLACE|TRUNCATE|ATTACH|DETACH|PRAGMA)\b',re.I)

def validate_sql(sql):
    sql=sql.strip().rstrip(';').strip()
    if not sql: return False,'Empty SQL query.'
    if not sql.upper().startswith('SELECT'): return False,'Only SELECT queries are allowed.'
    if FORBIDDEN_SQL.search(sql): return False,'Potentially destructive or administrative SQL detected.'
    if ';' in sql: return False,'Multiple SQL statements are not allowed.'
    return True,sql

def execute_read_only_sql(sql,max_rows=100):
    ok,result=validate_sql(sql)
    if not ok: return {'error':result}
    try:
        cur=conn.execute(result)
        rows=cur.fetchmany(max_rows)
        cols=[d[0] for d in cur.description] if cur.description else []
        return {'columns':cols,'rows':[list(r) for r in rows],'row_count_returned':len(rows)}
    except Exception as e:
        return {'error':str(e)}

print(execute_read_only_sql('SELECT * FROM customers LIMIT 3'))

## 🧠 4. Generate SQL with the LLM

In [ ]:
SYSTEM_PROMPT='''You are a SQL generation assistant for a SQLite database.\nGenerate exactly one read-only SELECT query answering the user's question.\nUse only tables and columns in the supplied schema.\nNever generate INSERT, UPDATE, DELETE, DROP, ALTER, CREATE, PRAGMA, or other write/admin statements.\nReturn JSON with exactly two keys: sql and explanation.'''

def generate_sql(question,schema):
    prompt=f'Database schema:\n{json.dumps(schema,indent=2)}\n\nUser question:\n{question}'
    response=client.responses.create(model=MODEL,instructions=SYSTEM_PROMPT,input=prompt)
    text=response.output_text.strip()
    try: return json.loads(text)
    except json.JSONDecodeError:
        match=re.search(r'\{.*\}',text,re.S)
        if match: return json.loads(match.group(0))
        raise ValueError('Could not parse model output: '+text)

print(json.dumps(generate_sql('Which customers are from Chennai?',schema),indent=2))

## 🔄 5. Build the SQL Database AI Agent

In [ ]:
def run_sql_agent(question,verbose=True):
    generated=generate_sql(question,schema)
    sql=generated.get('sql','')
    if verbose:
        print('📝 Generated SQL:\n',sql)
        print('\n💡 SQL Explanation:\n',generated.get('explanation',''))

    result=execute_read_only_sql(sql)
    if 'error' in result: return {'question':question,'sql':sql,'error':result['error']}

    final_prompt=f'''Answer the user's question using only the SQL result. Do not invent values. If there are no rows, say no matching records were found.\n\nQuestion: {question}\nSQL: {sql}\nResult: {json.dumps(result)}'''
    response=client.responses.create(model=MODEL,input=final_prompt)
    return {'question':question,'sql':sql,'result':result,'answer':response.output_text.strip()}

result=run_sql_agent('Which customers are from Chennai?')
print('\n💬 Final Answer:\n',result['answer'])

## 🧪 6. Test Natural-Language Questions

In [ ]:
questions=[
    'What products are in the Accessories category?',
    'Which customer placed the most orders?',
    'What is the total quantity of Laptop Pro units ordered?',
    'Show the top 3 customers by total order value.'
]

for q in questions:
    print('\n'+'='*80) 
    print('👤',q)
    r=run_sql_agent(q,verbose=False)
    print('📝 SQL:',r.get('sql'))
    print('💬 Answer:',r.get('answer',r.get('error')))

## 🔐 7. Test SQL Safety

In [ ]:
unsafe=['DELETE FROM customers','DROP TABLE orders','UPDATE products SET price=0','PRAGMA table_info(customers)']
for sql in unsafe:
    print(sql,'->',execute_read_only_sql(sql))

## 🎓 Interview Concepts

**Text-to-SQL:** converts natural language into executable SQL.

**Schema awareness:** gives the LLM the available tables and columns so it can generate grounded queries.

**SQL validation:** prevents unrestricted execution of model-generated SQL.

**Agentic workflow:** the system generates a query, executes a tool, observes the result, and produces a final answer.

**Production security:** use a dedicated read-only database account, query timeouts, limits, auditing, and stronger SQL parsing/AST validation.

# 🚀 Future Enhancements

- PostgreSQL / MySQL support
- 🔁 Automatic SQL error correction
- 🧠 Schema retrieval for large databases
- 🔐 Read-only DB credentials
- ⏱️ Query timeout and row limits
- 📊 Charts generated from query results
- 💬 Streamlit or React chat UI
- 🌐 FastAPI backend
- 📈 Tracing and evaluation
- 🧪 Text-to-SQL benchmark tests

# 🎉 Conclusion

This notebook demonstrates a complete learning-oriented **SQL Database AI Agent**: natural language → LLM-generated SQL → safe execution → database result → natural-language explanation.